In [ ]:
import astropy.units as u
import astropy.coordinates as apycoords
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
hclu = pd.read_csv("/Volumes/travelpassport/litclusterdatabases/HR24/HR24_clusters.csv")
hmem = pd.read_hdf("/Volumes/travelpassport/litclusterdatabases/HR24/HR24_members.h5")


open_cond = (
    (hclu.dist16 < 500) & (hclu.CST > 5) & (hclu.Type == "o") & (hclu.RV.notna())
)
mg_cond = (hclu.dist16 < 500) & (hclu.CST > 10) & (hclu.Type == "m") & (hclu.RV.notna())
cond = open_cond | mg_cond

hclu = hclu.loc[cond].reset_index(drop=True).copy()

hmem = hmem.loc[hmem.Name.isin(hclu.Name)].reset_index(drop=True).copy()

In [98]:
def xyzuvw(ra, dec, distance, pmra, pmdec, radial_velocity):
    """
    Converts ICRS ra (deg), dec (deg), parallax (mas), pmra (mas/yr),
    pmdec (mas/yr), and radial velocity (km/s) to heliocentric
    Cartesian XYZ (pc) and UVW (km/s)

    Args:
        ra (array-like): Right ascension in degrees
        dec (array-like): Declination in degrees
        distance (array-like): Distance in parsecs
        pmra (array-like): Proper motion in right ascension (mas/yr)
        pmdec (array-like): Proper motion in declination (mas/yr)
        radial_velocity (array-like): Radial velocity in km/s

    Returns:
        numpy.ndarray : (n x 6) XYZUVW array
    """

    c = apycoords.SkyCoord(
        ra=ra * u.deg,
        dec=dec * u.deg,
        distance=distance * u.pc,
        pm_ra_cosdec=pmra * u.mas / u.yr,
        pm_dec=pmdec * u.mas / u.yr,
        radial_velocity=radial_velocity * u.km / u.s,
        frame="icrs",
    )

    cg = c.transform_to(apycoords.Galactic())
    cg.representation_type = "cartesian"

    xyz = cg.cartesian.xyz.value.T.reshape(-1, 3)
    uvw = np.vstack([cg.U.value, cg.V.value, cg.W.value]).T
    xyzuvw = np.concatenate([xyz, uvw], axis=1)

    return xyzuvw

In [154]:
def galcen_cyl(ra, dec, distance, pmra, pmdec, radial_velocity):
    """
    Converts ICRS ra (deg), dec (deg), parallax (mas), pmra (mas/yr),
    pmdec (mas/yr), and radial velocity (km/s) to Galactocentric cylindrical coordinates
    rho (kpc), phi (deg), z (pc), v_rho (km/s), v_phi (km/s), v_z (km/s)

    LEFT-HANDED

    Args:
        ra (array-like): Right ascension in degrees
        dec (array-like): Declination in degrees
        distance (array-like): Distance in parsecs
        pmra (array-like): Proper motion in right ascension (mas/yr)
        pmdec (array-like): Proper motion in declination (mas/yr)
        radial_velocity (array-like): Radial velocity in km/s

    Returns:
        numpy.ndarray: (n x 6) array of galactocentric cylindrical coordinates


    """

    c = apycoords.SkyCoord(
        ra=ra * u.deg,
        dec=dec * u.deg,
        distance=distance * u.pc,
        pm_ra_cosdec=pmra * u.mas / u.yr,
        pm_dec=pmdec * u.mas / u.yr,
        radial_velocity=radial_velocity * u.km / u.s,
        frame="icrs",
    )

    v_sun = apycoords.CartesianDifferential([11.1, 245.0, 7.25] * u.km / u.s)
    gc_frame = apycoords.Galactocentric(
        galcen_distance=8.25 * u.kpc, z_sun=20.8 * u.pc, galcen_v_sun=v_sun
    )

    gc = c.transform_to(gc_frame)
    gc.representation_type = "cylindrical"

    cyl_coord = np.vstack(
        [
            gc.rho.to(u.kpc).value,
            180 - gc.phi.degree,
            gc.z.to(u.pc).value,
            gc.d_rho.to(u.km / u.s).value,
            -(gc.d_phi * gc.rho)
            .to(u.km / u.s, equivalencies=u.dimensionless_angles())
            .value,
            gc.d_z.to(u.km / u.s).value,
        ]
    ).T

    return cyl_coord

In [155]:
foo = xyzuvw(
    hclu.RA_ICRS.values,
    hclu.DE_ICRS.values,
    hclu.dist50.values,
    hclu.pmRA.values,
    hclu.pmDE.values,
    hclu.RV.values,
)

In [138]:
foo.shape

(288, 6)

In [139]:
import duckdb

In [140]:
df = duckdb.sql("""

    SELECT *

    FROM read_parquet('/Volumes/travelpassport/tables/allskylitjoin/sky_00000074.parquet')

    LIMIT 1000

""").df()

In [142]:
df.columns

Index(['source_id', 'ra', 'ra_error', 'dec', 'dec_error', 'parallax',
       'parallax_error', 'parallax_over_error', 'pm', 'pmra', 'pmra_error',
       'pmdec', 'pmdec_error', 'ruwe', 'phot_g_mean_flux',
       'phot_g_mean_flux_error', 'phot_g_mean_mag', 'phot_bp_mean_flux',
       'phot_bp_mean_flux_error', 'phot_bp_mean_mag', 'phot_rp_mean_flux',
       'phot_rp_mean_flux_error', 'phot_rp_mean_mag',
       'phot_bp_rp_excess_factor', 'bp_rp', 'bp_g', 'g_rp', 'radial_velocity',
       'radial_velocity_error', 'l', 'b', 'has_xp_continuous', 'has_rvs', 'X',
       'Y', 'Z', 'U', 'V', 'W', '__index_level_0__', 'source_id_1',
       'ocmg_name_1_SPYGLASS', 'ocmg_name_2_SPYGLASS', 'ocmg_name_3_SPYGLASS',
       'mem_prob_1_SPYGLASS', 'mem_prob_2_SPYGLASS', 'mem_prob_3_SPYGLASS',
       'ocmg_name_1_KCS20', 'ocmg_name_2_KCS20', 'ocmg_name_1_CG20',
       'ocmg_name_2_CG20', 'mem_prob_1_CG20', 'mem_prob_2_CG20',
       'ocmg_name_1_HR24', 'ocmg_name_2_HR24', 'ocmg_name_3_HR24',
       'mem

In [157]:
df[["new_x", "new_y", "new_z", "new_u", "new_v", "new_w"]] = xyzuvw(
    df.ra.values,
    df.dec.values,
    1000 / df.parallax.values,
    df.pmra.values,
    df.pmdec.values,
    df.radial_velocity.values,
)

In [163]:
df.new_w - df.W

0      NaN
1      NaN
2      NaN
3      NaN
4      NaN
      ... 
995    NaN
996    0.0
997    NaN
998    NaN
999    NaN
Length: 1000, dtype: float64